In [ ]:
import io
import pandas as pd
import paramiko

# --- 提供いただいたヘッダー見本（タブ区切りテキスト） ---
raw_header_text = """Timestamp (end of interval)	Int Temp (ｰC)	Ext Temp (ｰC)	Pressure (hPa)	Rel Humidity (%)	Wiper count	Vbatt (V)	40m Wind Speed (m/s)	40m Wind Speed Dispersion (m/s)	40m Wind Speed min (m/s)	40m Wind Speed max (m/s)	40m Wind Direction (ｰ)	40m Z-wind (m/s)	40m Z-wind Dispersion (m/s)	40m CNR (dB)	40m CNR min (dB)	40m Dopp Spect Broad (m/s)	40m Data Availability (%)		60m Wind Speed (m/s)	60m Wind Speed Dispersion (m/s)	60m Wind Speed min (m/s)	60m Wind Speed max (m/s)	60m Wind Direction (ｰ)	60m Z-wind (m/s)	60m Z-wind Dispersion (m/s)	60m CNR (dB)	60m CNR min (dB)	60m Dopp Spect Broad (m/s)	60m Data Availability (%)		80m Wind Speed (m/s)	80m Wind Speed Dispersion (m/s)	80m Wind Speed min (m/s)	80m Wind Speed max (m/s)	80m Wind Direction (ｰ)	80m Z-wind (m/s)	80m Z-wind Dispersion (m/s)	80m CNR (dB)	80m CNR min (dB)	80m Dopp Spect Broad (m/s)	80m Data Availability (%)		100m Wind Speed (m/s)	100m Wind Speed Dispersion (m/s)	100m Wind Speed min (m/s)	100m Wind Speed max (m/s)	100m Wind Direction (ｰ)	100m Z-wind (m/s)	100m Z-wind Dispersion (m/s)	100m CNR (dB)	100m CNR min (dB)	100m Dopp Spect Broad (m/s)	100m Data Availability (%)		120m Wind Speed (m/s)	120m Wind Speed Dispersion (m/s)	120m Wind Speed min (m/s)	120m Wind Speed max (m/s)	120m Wind Direction (ｰ)	120m Z-wind (m/s)	120m Z-wind Dispersion (m/s)	120m CNR (dB)	120m CNR min (dB)	120m Dopp Spect Broad (m/s)	120m Data Availability (%)		140m Wind Speed (m/s)	140m Wind Speed Dispersion (m/s)	140m Wind Speed min (m/s)	140m Wind Speed max (m/s)	140m Wind Direction (ｰ)	140m Z-wind (m/s)	140m Z-wind Dispersion (m/s)	140m CNR (dB)	140m CNR min (dB)	140m Dopp Spect Broad (m/s)	140m Data Availability (%)		160m Wind Speed (m/s)	160m Wind Speed Dispersion (m/s)	160m Wind Speed min (m/s)	160m Wind Speed max (m/s)	160m Wind Direction (ｰ)	160m Z-wind (m/s)	160m Z-wind Dispersion (m/s)	160m CNR (dB)	160m CNR min (dB)	160m Dopp Spect Broad (m/s)	160m Data Availability (%)		180m Wind Speed (m/s)	180m Wind Speed Dispersion (m/s)	180m Wind Speed min (m/s)	180m Wind Speed max (m/s)	180m Wind Direction (ｰ)	180m Z-wind (m/s)	180m Z-wind Dispersion (m/s)	180m CNR (dB)	180m CNR min (dB)	180m Dopp Spect Broad (m/s)	180m Data Availability (%)		200m Wind Speed (m/s)	200m Wind Speed Dispersion (m/s)	200m Wind Speed min (m/s)	200m Wind Speed max (m/s)	200m Wind Direction (ｰ)	200m Z-wind (m/s)	200m Z-wind Dispersion (m/s)	200m CNR (dB)	200m CNR min (dB)	200m Dopp Spect Broad (m/s)	200m Data Availability (%)		220m Wind Speed (m/s)	220m Wind Speed Dispersion (m/s)	220m Wind Speed min (m/s)	220m Wind Speed max (m/s)	220m Wind Direction (ｰ)	220m Z-wind (m/s)	220m Z-wind Dispersion (m/s)	220m CNR (dB)	220m CNR min (dB)	220m Dopp Spect Broad (m/s)	220m Data Availability (%)		240m Wind Speed (m/s)	240m Wind Speed Dispersion (m/s)	240m Wind Speed min (m/s)	240m Wind Speed max (m/s)	240m Wind Direction (ｰ)	240m Z-wind (m/s)	240m Z-wind Dispersion (m/s)	240m CNR (dB)	240m CNR min (dB)	240m Dopp Spect Broad (m/s)	240m Data Availability (%)		260m Wind Speed (m/s)	260m Wind Speed Dispersion (m/s)	260m Wind Speed min (m/s)	260m Wind Speed max (m/s)	260m Wind Direction (ｰ)	260m Z-wind (m/s)	260m Z-wind Dispersion (m/s)	260m CNR (dB)	260m CNR min (dB)	260m Dopp Spect Broad (m/s)	260m Data Availability (%)		280m Wind Speed (m/s)	280m Wind Speed Dispersion (m/s)	280m Wind Speed min (m/s)	280m Wind Speed max (m/s)	280m Wind Direction (ｰ)	280m Z-wind (m/s)	280m Z-wind Dispersion (m/s)	280m CNR (dB)	280m CNR min (dB)	280m Dopp Spect Broad (m/s)	280m Data Availability (%)		300m Wind Speed (m/s)	300m Wind Speed Dispersion (m/s)	300m Wind Speed min (m/s)	300m Wind Speed max (m/s)	300m Wind Direction (ｰ)	300m Z-wind (m/s)	300m Z-wind Dispersion (m/s)	300m CNR (dB)	300m CNR min (dB)	300m Dopp Spect Broad (m/s)	300m Data Availability (%)"""

# タブで分割し、前後スペースをカットした上で、「完全に空の要素」を排除する
clean_headers = [c.strip() for c in raw_header_text.split("\t") if c.strip() != ""]

sftp_host = "172.16.34.223"
sftp_user = "hayashi"
sftp_pass = "hayashi"
remote_path = "/home/hayashi/kaiseki/SampleData/FREA_LiDAR/WLS71045_sta_files/WLS71045_2025_02_01__00_10_00.sta"

ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(sftp_host, username=sftp_user, password=sftp_pass)
sftp = ssh.open_sftp()

try:
    print("サーバーからファイル全体をダウンロード中...")
    with sftp.open(remote_path, "rb") as f:
        binary_data = f.read()

    text_data = binary_data.decode("shift_jis", errors="ignore")

    print("データ部分（43行目以降）をスペース区切りで仮読み込み中...")
    # pandasの自動合体に頼らず、まずは全列をインデックス（0,1,2...）の表としてバラバラに読み込む
    df_raw = pd.read_csv(
        io.StringIO(text_data),
        sep="\s+",
        skiprows=42,
        header=None,
    )

    print("日付（1列目）と時刻（2列目）を1つのシリーズとして結合中...")
    # 0列目と1列目を文字列結合して日時オブジェクトに変換
    combined_datetime = pd.to_datetime(
        df_raw[0].astype(str) + " " + df_raw[1].astype(str), errors="coerce"
    )

    # 新しいデータフレームを作成。1列目を合体した日時にする
    df_clean = pd.DataFrame()
    df_clean[0] = combined_datetime

    # 元のデータの「3列目（インデックス2）」以降を、右側にガチャンと結合する
    # これにより日付・時刻の2列が綺麗に1列分にまとまります
    df_data_part = df_raw.iloc[:, 2:]
    df_clean = pd.concat([df_clean, df_data_part], axis=1)

    # 列番号を 0, 1, 2... にリセット
    df_clean.columns = range(df_clean.shape[1])

    # 実際のデータ列数と、クレンジングした見本ヘッダーの数を確認
    actual_cols = df_clean.shape[1]
    print(f"日時合体後のデータ実際の列数: {actual_cols} 列")
    print(f"見本リストの有効なヘッダー数: {len(clean_headers)} 個")

    # 列名を安全にマッピング
    final_columns = []
    for i in range(actual_cols):
        if i < len(clean_headers):
            final_columns.append(clean_headers[i])
        else:
            final_columns.append(f"Extra_Column_{i}")

    df_clean.columns = final_columns

    print("\n🎉 読み込み・整形が完全に成功しました！")
    print("最終データ形状（行数, 列数）:", df_clean.shape)

    print("\n--- 【確認】先頭5列の名前 ---")
    print(df_clean.columns[:5].tolist())

    print("\n--- 【確認】一番右端（末尾）の5列の名前 ---")
    print(df_clean.columns[-5:].tolist())

    print("\n--- データの最初の3行 ---")
    print(df_clean.head(3))

except Exception as e:
    print(f"\n❌ エラーが発生しました: {e}")

finally:
    sftp.close()
    ssh.close()

--- ファイルの先頭10行（Shift_JIS形式）を表示します ---
行 1: 'HeaderSize=40\n'
行 2: 'Version=WF 2.3.0.20432\n'
行 3: 'ID System=WLS71045\n'
行 4: 'ID Client=107764-1051162\n'
行 5: 'Location=FREA\n'
行 6: 'GPS Location=Lat:37.450597ｰN, Long:140.314023ｰE\n'
行 7: 'Comments=Campaign name: FREA 2024&Starting date: 14 June 2024\n'
行 8: 'FCR Option= OFF\n'
行 9: 'timezone=UTC+9\n'
行 10: '********************\n'


In [ ]:
＃読み込むだけの方
import io
import pandas as pd
import paramiko

sftp_host = "172.16.34.223"
sftp_user = "hayashi"
sftp_pass = "hayashi"
remote_path = "/home/hayashi/kaiseki/SampleData/FREA_LiDAR/WLS71045_sta_files/WLS71045_2025_02_01__00_10_00.sta"

ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(sftp_host, username=sftp_user, password=sftp_pass)
sftp = ssh.open_sftp()

try:
    print("サーバーからファイル全体をダウンロード中...")
    with sftp.open(remote_path, "rb") as f:
        binary_data = f.read()

    text_data = binary_data.decode("shift_jis", errors="ignore")

    print("ファイルを先頭行からそのまま読み込み中...")
    # スキップなし・ヘッダーなしで、1行目からすべて読み込む
    df_raw = pd.read_csv(
        io.StringIO(text_data),
        sep="\s+",
        header=None,
    )

    print("\n🎉 全データの読み込みが完了しました！")
    print("データ形状（行数, 列数）:", df_raw.shape)

    print("\n--- データの最初の10行 ---")
    print(df_raw.head(10))

except Exception as e:
    print(f"\n❌ エラーが発生しました: {e}")

finally:
    sftp.close()
    ssh.close()